<a href="https://colab.research.google.com/github/RafaelaMlucca/conformal-violence-prediction/blob/main/01_setup_dados_(3).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 01 — Setup dos dados (versão esparsa)

Carrega os dados já processados do projeto anterior e prepara três partições.

**Mudança importante:** mantenho as matrizes em formato esparso o tempo todo.
Converter para denso estoura a RAM do Colab grátis quando trabalhamos com a
calibração e o teste completos. Esparso ocupa ~10% da memória e o sklearn,
MAPIE e árvores de decisão aceitam direto sem conversão.

## 1. Configuração

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import json
import pickle
from scipy.sparse import load_npz, save_npz
from sklearn.model_selection import train_test_split

DRIVE = Path('/content/drive/MyDrive/projeto_violencia_mulher')

SAIDA = DRIVE / 'conformal_violence'
SAIDA.mkdir(exist_ok=True)

RANDOM_STATE = 42
ALVOS = ['y_fisic', 'y_psico', 'y_sexu']

## 2. Carregando os dados já processados

In [4]:
# 1. Remonta o Drive se necessário
from google.colab import drive
try:
    drive.mount('/content/drive', force_remount=True)
except:
    pass

# 2. Verifica se os arquivos existem no Drive
from pathlib import Path
SAIDA = Path('/content/drive/MyDrive/projeto_violencia_mulher/conformal_violence')
print('Arquivos no Drive:')
for arq in sorted(SAIDA.iterdir()):
    if arq.is_file():
        print(f'  {arq.name}: {arq.stat().st_size / 1024 / 1024:.1f} MB')

Mounted at /content/drive
Arquivos no Drive:
  cobertura_local_y_fisic.csv: 0.0 MB
  cobertura_local_y_psico.csv: 0.0 MB
  cobertura_local_y_sexu.csv: 0.0 MB
  dados_conformal.pkl: 3425.5 MB
  resultados_baseline.pkl: 9.8 MB
  resultados_locart.pkl: 24.4 MB
  tabela_comparativa_final.csv: 0.0 MB


In [5]:
X_train_orig = load_npz(DRIVE / 'X_train.npz')
X_test_orig = load_npz(DRIVE / 'X_test.npz')
y_train_orig = pd.read_parquet(DRIVE / 'y_train.parquet')
y_test_orig = pd.read_parquet(DRIVE / 'y_test.parquet')

with open(DRIVE / 'feature_names.json') as f:
    feature_names = json.load(f)

print(f'Treino original: {X_train_orig.shape[0]:,} linhas')
print(f'Teste:           {X_test_orig.shape[0]:,} linhas')

Treino original: 2,556,340 linhas
Teste:           875,656 linhas


## 3. Removendo casos com alvo faltante

Cerca de 3% das notificações têm pelo menos um dos três tipos sem classificação.
Removo para garantir comparação justa entre os três desfechos.

In [6]:
mask_train = y_train_orig[ALVOS].notna().all(axis=1).values
X_train_full = X_train_orig[mask_train]
y_train_full = y_train_orig[mask_train].reset_index(drop=True)

mask_test = y_test_orig[ALVOS].notna().all(axis=1).values
X_test_full = X_test_orig[mask_test]
y_test_full = y_test_orig[mask_test].reset_index(drop=True)

# converte float32 para int agora que não tem NaN
for alvo in ALVOS:
    y_train_full[alvo] = y_train_full[alvo].astype(int)
    y_test_full[alvo] = y_test_full[alvo].astype(int)

print(f'Treino após limpeza: {X_train_full.shape[0]:,} ({mask_train.mean():.1%})')
print(f'Teste após limpeza:  {X_test_full.shape[0]:,} ({mask_test.mean():.1%})')

Treino após limpeza: 2,476,734 (96.9%)
Teste após limpeza:  852,044 (97.3%)


## 4. Dividindo o treino em treino-puro e calibração

In [13]:
idx_all = np.arange(X_train_full.shape[0])

idx_train, idx_cal = train_test_split(
    idx_all,
    test_size=0.30,
    stratify=y_train_full['y_sexu'].values,
    random_state=RANDOM_STATE
)

print(f'Treino:    {len(idx_train):,}')
print(f'Calibração: {len(idx_cal):,}')

Treino:    1,733,713
Calibração: 743,021


## 5. Reduzindo o treino para 80 mil

Treino reduzido só para acelerar o ajuste da logística. Calibração e teste
ficam inteiros porque agora estão esparsos e cabem na memória.

In [14]:
N_TREINO = 900_000

if len(idx_train) > N_TREINO:
    idx_train, _ = train_test_split(
        idx_train,
        train_size=N_TREINO,
        stratify=y_train_full.iloc[idx_train]['y_sexu'].values,
        random_state=RANDOM_STATE
    )

print(f'Treino final: {len(idx_train):,}')

Treino final: 900,000


## 6. Montando os splits

**Sem converter para denso.** As matrizes ficam em formato esparso CSR.

In [15]:
# fatia as matrizes esparsas — continua esparso
X_train = X_train_full[idx_train].tocsr()
X_cal   = X_train_full[idx_cal].tocsr()
X_test  = X_test_full.tocsr()

y_train = y_train_full.iloc[idx_train].reset_index(drop=True)
y_cal   = y_train_full.iloc[idx_cal].reset_index(drop=True)
y_test  = y_test_full.copy()

# memória usada pelas matrizes esparsas (estimativa)
def mb_esparsa(M):
    return (M.data.nbytes + M.indices.nbytes + M.indptr.nbytes) / 1024 / 1024

print(f'Memória das matrizes esparsas:')
print(f'  X_train: {mb_esparsa(X_train):.1f} MB')
print(f'  X_cal:   {mb_esparsa(X_cal):.1f} MB')
print(f'  X_test:  {mb_esparsa(X_test):.1f} MB')

print('\nPrevalências:')
print(f'{"alvo":<12} {"treino":<10} {"calibração":<12} {"teste":<10}')
for alvo in ALVOS:
    p_tr = y_train[alvo].mean()
    p_cal = y_cal[alvo].mean()
    p_te = y_test[alvo].mean()
    print(f'{alvo:<12} {p_tr:<10.1%} {p_cal:<12.1%} {p_te:<10.1%}')

Memória das matrizes esparsas:
  X_train: 559.6 MB
  X_cal:   462.0 MB
  X_test:  529.8 MB

Prevalências:
alvo         treino     calibração   teste     
y_fisic      50.7%      50.8%        47.1%     
y_psico      23.4%      23.4%        21.9%     
y_sexu       15.8%      15.8%        16.9%     


## 7. Salvando

Salvo as matrizes esparsas em arquivos `.npz` separados e os `y` em parquet.
Isso é mais eficiente que pickle e os arquivos ficam bem menores.

In [16]:
save_npz(SAIDA / 'X_train.npz', X_train)
save_npz(SAIDA / 'X_cal.npz', X_cal)
save_npz(SAIDA / 'X_test.npz', X_test)

y_train.to_parquet(SAIDA / 'y_train.parquet', index=False)
y_cal.to_parquet(SAIDA / 'y_cal.parquet', index=False)
y_test.to_parquet(SAIDA / 'y_test.parquet', index=False)

# metadados
metadados = {
    'feature_names': feature_names,
    'alvos': ALVOS,
    'shapes': {
        'X_train': list(X_train.shape),
        'X_cal':   list(X_cal.shape),
        'X_test':  list(X_test.shape),
    }
}
with open(SAIDA / 'metadados.json', 'w') as f:
    json.dump(metadados, f, indent=2)

print(f'Salvo em {SAIDA}')
for arq in sorted(SAIDA.iterdir()):
    if arq.is_file():
        print(f'  {arq.name}: {arq.stat().st_size / 1024 / 1024:.1f} MB')

Salvo em /content/drive/MyDrive/projeto_violencia_mulher/conformal_violence
  X_cal.npz: 15.0 MB
  X_test.npz: 15.7 MB
  X_train.npz: 18.2 MB
  cobertura_local_y_fisic.csv: 0.0 MB
  cobertura_local_y_psico.csv: 0.0 MB
  cobertura_local_y_sexu.csv: 0.0 MB
  dados_conformal.pkl: 3425.5 MB
  metadados.json: 0.0 MB
  resultados_baseline.pkl: 9.8 MB
  resultados_locart.pkl: 24.4 MB
  tabela_comparativa_final.csv: 0.0 MB
  y_cal.parquet: 0.3 MB
  y_test.parquet: 0.3 MB
  y_train.parquet: 0.4 MB


## Próximo passo

Notebook 02: MAPIE baseline.